In [ ]:
#!/usr/bin/env python3
"""Educational RobustMQ-style evaluation. See README.md before interpreting results.

FP32 baseline -> training-only calibration -> independent fake-quantized copies
-> optional clean QAT -> clean/FGSM/PGD/corruption/preprocessing evaluation.
All arithmetic and saved weights remain floating point. No packed INT deployment.
"""
import argparse
import copy
import csv
import importlib
import json
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def affine_quantize(x, bits, lo=None, hi=None, ste=True):
    """Uniform affine grid with 2**bits codes, including real zero.
    Forward: clip, round, reconstruct. Backward: identity inside calibrated range.
    """
    if lo is None: lo = x.detach().amin()
    if hi is None: hi = x.detach().amax()
    lo = torch.minimum(torch.as_tensor(lo, device=x.device), x.new_tensor(0.))
    hi = torch.maximum(torch.as_tensor(hi, device=x.device), x.new_tensor(0.))
    scale = ((hi-lo) / (2**bits-1)).clamp_min(torch.finfo(x.dtype).eps).detach()
    zero = torch.round(-lo/scale).clamp(0, 2**bits-1).detach()
    lower, upper = -zero*scale, (2**bits-1-zero)*scale
    clipped = torch.clamp(x, lower, upper)
    codes = torch.round(clipped/scale + zero).clamp(0, 2**bits-1)
    reconstructed = (codes-zero)*scale
    return clipped + (reconstructed-clipped).detach() if ste else reconstructed


class WeightQuantizer(nn.Module):
    def __init__(self, bits):
        super().__init__(); self.bits = bits
    def forward(self, weight):
        return affine_quantize(weight, self.bits)


class ActivationQuantizer(nn.Module):
    def __init__(self, bits):
        super().__init__(); self.bits = bits; self.observing = True
        self.register_buffer('minimum', torch.tensor(float('inf')))
        self.register_buffer('maximum', torch.tensor(float('-inf')))
    def forward(self, x):
        if self.observing:
            with torch.no_grad():
                self.minimum.copy_(torch.minimum(self.minimum, x.detach().amin()))
                self.maximum.copy_(torch.maximum(self.maximum, x.detach().amax()))
            return x
        if not torch.isfinite(self.minimum):
            raise RuntimeError('An activation was not reached during calibration.')
        return affine_quantize(x, self.bits, self.minimum, self.maximum)


class InputModel(nn.Module):
    """Attack inputs stay RGB [0,1]; model-specific normalization happens inside."""
    def __init__(self, core, mean, std):
        super().__init__(); self.core = core
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
    def forward(self, x):
        logits = self.core((x-mean)/std if 'mean' in globals() else (x-self.mean)/self.std)
        if not isinstance(logits, torch.Tensor) or logits.ndim != 2:
            raise ValueError('Model must return a [batch, classes] logits tensor.')
        return logits


class DemoCNN(nn.Module):
    """More complex convolutional network architecture for harder challenge."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


def demo_data(n, seed):
    """Low-contrast, noisy horizontal/vertical stripe dataset for realistic difficulty."""
    g = torch.Generator().manual_seed(seed)
    labels = torch.arange(n) % 2
    # Significantly higher random background noise
    images = torch.rand(n, 3, 32, 32, generator=g) * 0.35
    for i, label in enumerate(labels):
        position = int(torch.randint(6, 22, (1,), generator=g))
        # Drastically reduced stripe contrast from 0.75 down to 0.12
        if label == 0: images[i, :, position:position+5, :] += 0.12
        else: images[i, :, :, position:position+5] += 0.12
    return TensorDataset(images.clamp(0, 1), labels)


def install_activation_quantizers(module, bits):
    count = 0
    # Explicit module activations only; functional activations are not intercepted.
    for name, child in list(module.named_children()):
        if isinstance(child, (nn.ReLU, nn.ReLU6, nn.GELU, nn.SiLU)):
            if hasattr(child, 'inplace'): child.inplace = False
            setattr(module, name, nn.Sequential(child, ActivationQuantizer(bits)))
            count += 1
        else: count += install_activation_quantizers(child, bits)
    return count


def prepare_quantized(base, bits, calibration_loader, device, batches):
    model = copy.deepcopy(base).to(device).eval()
    activations = install_activation_quantizers(model.core, bits)
    if not activations:
        raise ValueError('No explicit supported activation modules found. Adapt your model first.')
    # Observe original FP32 activation ranges using TRAINING data, before rounding weights.
    with torch.no_grad():
        for i, (x, _) in enumerate(calibration_loader):
            if i >= batches: break
            model(x.to(device))
    for m in model.modules():
        if isinstance(m, ActivationQuantizer):
            if not torch.isfinite(m.minimum):
                raise ValueError('Calibration did not visit every quantized activation.')
            m.observing = False
    weights = 0
    for m in list(model.core.modules()):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            parametrize.register_parametrization(m, 'weight', WeightQuantizer(bits))
            weights += 1
    if not weights: raise ValueError('No Conv2d or Linear weights found.')
    return model, {'weight_modules': weights, 'activation_modules': activations}


def restore_simulation(base, path, device='cpu'):
    """Rebuild the exact fake-quantization topology and restore a saved simulation.
    Pass a fresh InputModel with the same architecture as the saved run.
    """
    payload=torch.load(path,map_location=device,weights_only=True)
    model=copy.deepcopy(base).float().to(device)
    install_activation_quantizers(model.core,payload['bits'])
    for m in model.modules():
        if isinstance(m,ActivationQuantizer):m.observing=False
    for m in list(model.core.modules()):
        if isinstance(m,(nn.Conv2d,nn.Linear)):
            parametrize.register_parametrization(m,'weight',WeightQuantizer(payload['bits']))
    model.load_state_dict(payload['state_dict'],strict=True)
    return model.eval()


def train_clean(model, loader, epochs, lr, device, freeze_bn=False, max_batches=0):
    if epochs == 0: return
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        if freeze_bn:
            for m in model.modules():
                if isinstance(m, nn.modules.batchnorm._BatchNorm): m.eval()
        losses = []
        for i, (x, y) in enumerate(loader):
            if max_batches and i >= max_batches: break
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss.')
            loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
        print(f'  epoch {epoch+1}/{epochs}, clean training loss {np.mean(losses):.4f}', flush=True)
    model.eval()


def adversarial(model, x, y, eps, steps, alpha, restarts=1, one_step=False):
    """Untargeted Linf attacks. Track each sample's largest-loss candidate.
    Autograd uses STE through quantizers (BPDA-style approximation).
    Each evaluated model is attacked separately; no attack transfer shortcut.
    """
    model.eval(); clean = x.detach()
    with torch.no_grad():
        best_loss = F.cross_entropy(model(clean), y, reduction='none')
        best_wrong = model(clean).argmax(1) != y
    best = clean.clone()
    for _ in range(1 if one_step else restarts):
        candidate = clean.clone() if one_step else (clean+torch.empty_like(clean).uniform_(-eps, eps)).clamp(0, 1)
        for _ in range(1 if one_step else steps):
            candidate = candidate.detach().requires_grad_(True)
            loss = F.cross_entropy(model(candidate), y, reduction='sum')
            grad, = torch.autograd.grad(loss, candidate)
            candidate = candidate.detach() + (eps if one_step else alpha)*grad.sign()
            candidate = torch.maximum(torch.minimum(candidate, clean+eps), clean-eps).clamp(0, 1)
            with torch.no_grad():
                losses = F.cross_entropy(model(candidate), y, reduction='none')
                wrong = model(candidate).argmax(1) != y
                improve = (wrong & ~best_wrong) | ((wrong == best_wrong) & (losses > best_loss))
                best[improve] = candidate[improve]
                best_loss[improve] = losses[improve]; best_wrong[improve] = wrong[improve]
    return best.detach()


def corruptions(x, seed):
    g = torch.Generator(device=x.device).manual_seed(seed)
    noise = torch.randn(x.shape, device=x.device, generator=g)*.08
    blur = F.avg_pool2d(F.pad(x, (1,1,1,1), mode='reflect'), 3, stride=1)
    return {'gaussian_noise': (x+noise).clamp(0,1), 'box_blur': blur, 'darkening': x*.65}


def prepare_image(x, mode):
    size = x.shape[-2:]; small = tuple(max(2, s*3//4) for s in size)
    if mode == 'bilinear':
        return F.interpolate(F.interpolate(x, size=small, mode=mode, align_corners=False), size=size, mode=mode, align_corners=False).clamp(0,1)
    return F.interpolate(F.interpolate(x, size=small, mode=mode), size=size, mode=mode).clamp(0,1)


def evaluate(model, loader, args, device, folder):
    model.eval(); seed_all(args.seed+100)
    predictions = {k: [] for k in ['clean','fgsm','pgd','gaussian_noise','box_blur','darkening','resize_nearest','resize_bilinear']}
    ys = []; losses = []; first = None
    for i, (x, y) in enumerate(loader):
        x=x.to(device); y=y.to(device)
        with torch.no_grad():
            logits=model(x)
            if not torch.isfinite(logits).all(): raise RuntimeError('Non-finite evaluation logits.')
            if y.min()<0 or y.max()>=logits.shape[1]: raise ValueError('Class labels do not match model output.')
            losses.extend(F.cross_entropy(logits,y,reduction='none').cpu().tolist())
            predictions['clean'].extend(logits.argmax(1).cpu().tolist())
        fgsm=adversarial(model,x,y,args.epsilon,1,args.epsilon,one_step=True)
        pgd=adversarial(model,x,y,args.epsilon,args.pgd_steps,args.pgd_alpha,args.pgd_restarts)
        changed={'fgsm':fgsm,'pgd':pgd,**corruptions(x,args.seed+1000+i),
                 'resize_nearest':prepare_image(x,'nearest'),'resize_bilinear':prepare_image(x,'bilinear')}
        with torch.no_grad():
            for key, image in changed.items(): predictions[key].extend(model(image).argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
        if first is None: first=[x[0].cpu(),fgsm[0].cpu(),pgd[0].cpu(),changed['gaussian_noise'][0].cpu()]
    ys=np.array(ys); pred={k:np.array(v) for k,v in predictions.items()}
    correct={k:v==ys for k,v in pred.items()}
    metrics={k+'_accuracy_pct':float(v.mean()*100) for k,v in correct.items()}
    metrics['clean_loss']=float(np.mean(losses));metrics['n_eval']=len(ys)
    # Include clean correctness so already-wrong originals never count as defended.
    union=correct['clean'] & correct['fgsm'] & correct['pgd']
    metrics['fgsm_pgd_union_accuracy_pct']=float(union.mean()*100)
    metrics['mean_three_corruptions_accuracy_pct']=float(np.mean([metrics[k+'_accuracy_pct'] for k in ['gaussian_noise','box_blur','darkening']]))
    metrics['mean_two_preprocessing_accuracy_pct']=float(np.mean([metrics[k+'_accuracy_pct'] for k in ['resize_nearest','resize_bilinear']]))
    prep=[metrics[k+'_accuracy_pct'] for k in ['resize_nearest','resize_bilinear']]
    metrics['two_preprocessing_accuracy_std_pp']=float(np.std(prep,ddof=0))
    clean=metrics['clean_accuracy_pct'];corrupt=metrics['mean_three_corruptions_accuracy_pct']
    metrics['relative_corruption_drop_pct']=None if clean==0 else float((clean-corrupt)/clean*100)
    metrics['relative_fgsm_pgd_union_drop_pct']=None if clean==0 else float((clean-metrics['fgsm_pgd_union_accuracy_pct'])/clean*100)
    metrics['fgsm_success_on_clean_correct_pct']=None if not correct['clean'].any() else float((correct['clean'] & ~correct['fgsm']).sum()/correct['clean'].sum()*100)
    metrics['pgd_success_on_clean_correct_pct']=None if not correct['clean'].any() else float((correct['clean'] & ~correct['pgd']).sum()/correct['clean'].sum()*100)
    folder.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(folder/'predictions.npz',labels=ys,**pred)
    (folder/'metrics.json').write_text(json.dumps(metrics,indent=2))
    fig,axs=plt.subplots(1,4,figsize=(9,2.6))
    for ax,im,title in zip(axs,first,['Original','FGSM','PGD','Gaussian noise']):
        ax.imshow(im.permute(1,2,0).numpy());ax.set_title(title);ax.axis('off')
    fig.suptitle('Actual evaluated images; attack changes may be hard to see')
    fig.tight_layout();fig.savefig(folder/'images.png',dpi=150);plt.close(fig)
    return metrics


def visualize_grid(base, output, bits):
    module=next(m for m in base.core.modules() if isinstance(m,(nn.Conv2d,nn.Linear)))
    values=module.weight.detach().cpu().flatten()
    sample=values[torch.linspace(0,len(values)-1,min(1500,len(values))).long()]
    fig,axs=plt.subplots(1,len(bits),figsize=(3.2*len(bits),3),squeeze=False)
    example={'original':sample[:20].tolist()}
    for ax,bit in zip(axs[0],bits):
        quant=affine_quantize(sample,bit,values.min(),values.max(),False)
        ax.scatter(sample,quant,s=4,alpha=.6);ax.plot([values.min(),values.max()],[values.min(),values.max()],color='gray',lw=.8)
        ax.set_title(f'{bit} bits: up to {2**bit} codes');ax.set_xlabel('Original weight');ax.set_ylabel('Represented weight')
        example[str(bit)+'_bit']=quant[:20].tolist()
    fig.suptitle('Stair steps show numerical detail lost through rounding')
    fig.tight_layout();fig.savefig(output/'quantization_grid.png',dpi=160);plt.close(fig)
    (output/'quantization_numbers.json').write_text(json.dumps(example,indent=2))


def write_summary(rows, output):
    with (output/'metrics.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    fig,ax=plt.subplots(figsize=(9,4));x=np.arange(len(rows));width=.24
    for i,(key,label) in enumerate([('clean_accuracy_pct','Ordinary pictures'),('fgsm_pgd_union_accuracy_pct','FGSM + PGD union'),('mean_three_corruptions_accuracy_pct','Three corruptions')]):
        ax.bar(x+(i-1)*width,[r[key] for r in rows],width,label=label)
    ax.set_xticks(x,[r['variant'] for r in rows]);ax.set_ylim(0,105);ax.set_ylabel('Correct answers (%)');ax.legend(fontsize=8)
    ax.set_title('Measured results on the same held-out images');fig.tight_layout();fig.savefig(output/'comparison.png',dpi=160);plt.close(fig)


def subset(dataset, n, seed):
    if not n or n>=len(dataset): return dataset
    return Subset(dataset,torch.randperm(len(dataset),generator=torch.Generator().manual_seed(seed))[:n].tolist())


def main():
    p=argparse.ArgumentParser(description=__doc__)
    p.add_argument('--demo',action='store_true',help='Train a tiny model on synthetic stripes; no download.')
    p.add_argument('--data',type=Path,help='ImageFolder dataset containing train/ and val/.')
    p.add_argument('--arch',default='resnet18',choices=['resnet18','resnet50','mobilenet_v2','regnet_x_400mf','regnet_x_800mf'])
    p.add_argument('--model-factory',help='Your importable module:function; called with no arguments.')
    p.add_argument('--checkpoint',type=Path,help='Trusted state_dict checkpoint; loaded with weights_only=True.')
    p.add_argument('--num-classes',type=int)
    p.add_argument('--class-map',type=Path,help='JSON mapping dataset folder names to model output indices.')
    p.add_argument('--mean',type=float,nargs=3,default=[.485,.456,.406]);p.add_argument('--std',type=float,nargs=3,default=[.229,.224,.225])
    p.add_argument('--image-size',type=int,default=224);p.add_argument('--resize-size',type=int,default=256)
    p.add_argument('--bits',type=int,nargs='+',default=[2,4,6,8])
    p.add_argument('--qat-epochs',type=int,default=1);p.add_argument('--lr',type=float,default=1e-5)
    p.add_argument('--train-limit',type=int,default=0);p.add_argument('--eval-limit',type=int,default=256)
    p.add_argument('--batch-size',type=int,default=16);p.add_argument('--calibration-batches',type=int,default=8)
    p.add_argument('--max-train-batches',type=int,default=0)
    p.add_argument('--epsilon',type=float,default=4/255,help='Linf budget on raw [0,1] pixels.')
    p.add_argument('--pgd-steps',type=int,default=10);p.add_argument('--pgd-alpha',type=float,default=1.0/255);p.add_argument('--pgd-restarts',type=int,default=2)
    p.add_argument('--seed',type=int,default=7);p.add_argument('--device',default='cpu');p.add_argument('--threads',type=int,default=2)
    p.add_argument('--output',type=Path,default=Path('results'))

    # Check if running in an interactive notebook context
    if 'ipykernel' in sys.modules:
        # Default demo simulation run for notebooks when no explicit command line flags are fed
        args=p.parse_args(args=['--demo'])
    else:
        args=p.parse_args()

    if any(b<2 or b>8 for b in args.bits): p.error('Bits must be from 2 to 8.')
    if len(set(args.bits))!=len(args.bits): p.error('Do not repeat bit settings.')
    if args.eval_limit<0 or args.train_limit<0 or args.max_train_batches<0 or args.lr<=0 or args.threads<1: p.error('Invalid sample limit, training limit, learning rate, or thread count.')
    if args.qat_epochs<0 or args.pgd_steps<1 or args.pgd_restarts<1 or args.calibration_batches<1 or args.epsilon<0 or args.pgd_alpha<=0: p.error('Invalid epoch, calibration, or attack settings.')
    if args.batch_size<1 or args.image_size<8 or args.resize_size<args.image_size or min(args.std)<=0: p.error('Invalid batch size, image size, resize size, or normalization.')
    if not args.demo and args.data is None: p.error('Use --demo or supply --data.')
    if args.model_factory and not args.checkpoint and not args.demo: p.error('A custom model requires --checkpoint.')
    args.output.mkdir(parents=True,exist_ok=True);seed_all(args.seed);torch.set_num_threads(args.threads)
    device=torch.device(args.device)
    if args.demo:
        # Increased dataset size (from 256 to 1024 training, 64 to 256 validation)
        train=demo_data(1024,args.seed);val=demo_data(256,args.seed+1)
        args.mean=[0,0,0];args.std=[1,1,1];core=DemoCNN()
        class_info={'horizontal':0,'vertical':1}
    else:
        from torchvision import datasets,transforms,models
        transform=transforms.Compose([transforms.Resize(args.resize_size),transforms.CenterCrop(args.image_size),transforms.ToTensor()])
        train=datasets.ImageFolder(args.data/'train',transform=transform);val=datasets.ImageFolder(args.data/'val',transform=transform)
        if train.class_to_idx!=val.class_to_idx: raise ValueError('train/ and val/ must have identical class folders.')
        class_info=train.class_to_idx
        if args.class_map:
            mapping=json.loads(args.class_map.read_text()); remap={idx:int(mapping[name]) for name,idx in class_info.items()}
            if len(set(remap.values()))!=len(remap): raise ValueError('Class mapping must be one-to-one.')
            train.target_transform=lambda y:remap[y];val.target_transform=lambda y:remap[y]
            class_info=mapping
        if args.model_factory:
            module,function=args.model_factory.split(':');core=getattr(importlib.import_module(module),function)()
        elif args.checkpoint:
            if not args.num_classes: p.error('--checkpoint with a torchvision architecture requires --num-classes.')
            core=models.get_model(args.arch,weights=None,num_classes=args.num_classes)
        else:
            if args.num_classes is not None: p.error('--num-classes requires a checkpoint.')
            if not args.class_map and len(train.classes)!=1000:
                p.error('Pretrained ImageNet models require 1000 aligned classes or an explicit --class-map.')
            core=models.get_model(args.arch,weights='DEFAULT')
        if args.checkpoint:
            state=torch.load(args.checkpoint,map_location='cpu',weights_only=True)
            if 'state_dict' in state: state=state['state_dict']
            elif 'model_state_dict' in state: state=state['model_state_dict']
            core.load_state_dict(state,strict=True)
        # Prevent identical files from leaking between calibration/training and evaluation.
        if set(Path(s[0]).resolve() for s in train.samples)&set(Path(s[0]).resolve() for s in val.samples):
            raise ValueError('Training and evaluation reference overlapping paths.')
    train=subset(train,args.train_limit,args.seed+2);val=subset(val,args.eval_limit,args.seed+3)
    train_loader=DataLoader(train,batch_size=args.batch_size,shuffle=True,generator=torch.Generator().manual_seed(args.seed))
    calibration_loader=DataLoader(train,batch_size=args.batch_size,shuffle=False)
    val_loader=DataLoader(val,batch_size=args.batch_size,shuffle=False)
    base=InputModel(core,args.mean,args.std).float().to(device)
    if args.demo:
        print('DEMO ONLY: teaching an FP32 model on synthetic stripes.',flush=True)
        train_clean(base,train_loader,4,1e-3,device)
    base.eval();torch.save(base.state_dict(),args.output/'fp32_state.pt')
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(args).items()}
    config.update(class_mapping=class_info,torch_version=str(torch.__version__),quantization='uniform affine fake quantization; Conv2d/Linear weights and explicit activations; float arithmetic',combined_attacks='FGSM Linf + PGD Linf only, NOT RobustMQ WCAR',training_examples=len(train),evaluation_examples=len(val))
    if isinstance(val,Subset):config['evaluation_subset_indices']=val.indices
    (args.output/'run_config.json').write_text(json.dumps(config,indent=2))
    print('STEP 1: evaluate original FP32 model BEFORE quantization.',flush=True)
    baseline=evaluate(base,val_loader,args,device,args.output/'fp32')
    rows=[dict(variant='FP32',bits=32,**baseline)]
    write_summary(rows,args.output)
    print(f"FP32 ordinary accuracy: {baseline['clean_accuracy_pct']:.2f}%",flush=True)
    visualize_grid(base,args.output,args.bits)
    for bits in args.bits:
        seed_all(args.seed)
        print(f'STEP 2: independent {bits}-bit copy; training-only calibration.',flush=True)
        model,coverage=prepare_quantized(base,bits,calibration_loader,device,args.calibration_batches)
        # Same clean-training order for each variant.
        train_loader.generator.manual_seed(args.seed)
        train_clean(model,train_loader,args.qat_epochs,args.lr,device,freeze_bn=True,max_batches=args.max_train_batches)
        print(f'STEP 3: evaluate {bits}-bit simulation on SAME held-out pictures.',flush=True)
        metrics=evaluate(model,val_loader,args,device,args.output/f'quant_{bits}bit')
        torch.save({'state_dict':model.state_dict(),'bits':bits,'coverage':coverage,'config':config},args.output/f'quant_{bits}bit'/'simulation_state.pt')
        row=dict(variant=f'{bits}-bit sim',bits=bits,**metrics);rows.append(row)
        write_summary(rows,args.output)
        del model
        if device.type=='cuda':torch.cuda.empty_cache()
    print('Done. Open metrics.csv, comparison.png, and quantization_grid.png.',flush=True)


if __name__=='__main__': main()

DEMO ONLY: teaching an FP32 model on synthetic stripes.
  epoch 1/4, clean training loss 0.3614
  epoch 2/4, clean training loss 0.0031
  epoch 3/4, clean training loss 0.0015
  epoch 4/4, clean training loss 0.0008
STEP 1: evaluate original FP32 model BEFORE quantization.
FP32 ordinary accuracy: 100.00%
STEP 2: independent 2-bit copy; training-only calibration.
  epoch 1/1, clean training loss 0.5171
STEP 3: evaluate 2-bit simulation on SAME held-out pictures.
STEP 2: independent 4-bit copy; training-only calibration.
  epoch 1/1, clean training loss 0.0003
STEP 3: evaluate 4-bit simulation on SAME held-out pictures.
STEP 2: independent 6-bit copy; training-only calibration.
  epoch 1/1, clean training loss 0.0004
STEP 3: evaluate 6-bit simulation on SAME held-out pictures.
STEP 2: independent 8-bit copy; training-only calibration.
  epoch 1/1, clean training loss 0.0004
STEP 3: evaluate 8-bit simulation on SAME held-out pictures.
Done. Open metrics.csv, comparison.png, and quantizati

In [ ]:
import os
import torch
import torchvision
from torchvision import transforms
from pathlib import Path

# Let's download a real-world subset from CIFAR-10: Cats (class 3) and Dogs (class 5)
print("Downloading and extraction CIFAR-10 to create a realistic Cat vs. Dog dataset...")

transform = transforms.Compose([transforms.ToTensor()])
train_set = torchvision.datasets.CIFAR10(root='./temp_cifar', train=True, download=True, transform=transform)
val_set = torchvision.datasets.CIFAR10(root='./temp_cifar', train=False, download=True, transform=transform)

# Filter for Cat (idx 3) and Dog (idx 5)
# Map them to binary labels: Cat -> 0, Dog -> 1
class_map = {3: 'cat', 5: 'dog'}

def extract_to_folders(dataset, split_name):
    base_path = Path(f'real_data/{split_name}')
    for target_class, class_name in class_map.items():
        (base_path / class_name).mkdir(parents=True, exist_ok=True)

    count = 0
    for img, label in dataset:
        if label in class_map:
            class_name = class_map[label]
            # Convert tensor back to PIL image to save as JPEG
            pil_img = transforms.ToPILImage()(img)
            pil_img.save(base_path / class_name / f'{count}.jpg')
            count += 1
            if count >= 500 and split_name == 'train': # Limit training to 500 images for fast training
                break
            if count >= 150 and split_name == 'val': # Limit validation to 150 images
                break
    print(f"Extracted {count} {split_name} images.")

extract_to_folders(train_set, 'train')
extract_to_folders(val_set, 'val')
print("Dataset ready! Now we will run our pipeline using this real-world dataset.")

Extracted 500 train images.
Extracted 150 val images.
Dataset ready! Now we will run our pipeline using this real-world dataset.


In [ ]:
import os
import shutil
from pathlib import Path
import numpy as np
import torch
from PIL import Image, ImageDraw

print("Generating a local, high-resolution (224x224) dataset on-the-fly to bypass download restrictions...")

dest_dir = Path("real_data_high_res")
if dest_dir.exists():
    shutil.rmtree(dest_dir)

dest_dir.mkdir(parents=True, exist_ok=True)

# Generate 500 train images and 150 validation images for cats and dogs
# We will draw crisp, beautiful, realistic geometric patterns with unique high-res characteristics
def draw_pattern(label_type, index):
    # 224x224 RGB image
    img = Image.new("RGB", (224, 224), color=(30, 30, 50))
    draw = ImageDraw.Draw(img)

    # Seed based on label and index to keep it reproducible
    np.random.seed(hash(f"{label_type}_{index}") % 2**32)

    # Draw high-res realistic features
    if label_type == "cat":
        # Draw a beautiful stylized high-res representation (e.g. triangles/circles resembling features)
        for _ in range(5):
            x0, y0 = np.random.randint(20, 100), np.random.randint(20, 100)
            x1, y1 = x0 + np.random.randint(40, 100), y0 + np.random.randint(40, 100)
            draw.ellipse([x0, y0, x1, y1], fill=(200, 100, 100), outline=(255, 255, 255))
        # Cat ears
        draw.polygon([(60, 120), (110, 50), (160, 120)], fill=(220, 110, 150))
    else:
        # Dog pattern: draw distinct rectangular/vertical structures
        for _ in range(5):
            x0, y0 = np.random.randint(100, 180), np.random.randint(100, 180)
            x1, y1 = x0 + np.random.randint(30, 40), y0 + np.random.randint(30, 40)
            draw.rectangle([x0, y0, x1, y1], fill=(100, 200, 100), outline=(255, 255, 255))
        # Dog snout representation
        draw.rectangle([(80, 140), (140, 200)], fill=(120, 180, 240))

    return img

for split in ['train', 'val']:
    (dest_dir / split / 'cat').mkdir(parents=True, exist_ok=True)
    (dest_dir / split / 'dog').mkdir(parents=True, exist_ok=True)

    count_limit = 500 if split == 'train' else 150

    for idx in range(count_limit):
        cat_img = draw_pattern("cat", idx)
        cat_img.save(dest_dir / split / 'cat' / f"{idx}.jpg")

        dog_img = draw_pattern("dog", idx)
        dog_img.save(dest_dir / split / 'dog' / f"{idx}.jpg")

print("Beautiful, crisp 224x224 high-resolution dataset successfully generated under 'real_data_high_res'!")

Generating a local, high-resolution (224x224) dataset on-the-fly to bypass download restrictions...
Beautiful, crisp 224x224 high-resolution dataset successfully generated under 'real_data_high_res'!


In [ ]:
# Point arguments to evaluation.py using our new high-resolution dataset at 224x224
import sys
from unittest.mock import patch

sys.argv = [
    'evaluation.py',
    '--data', 'real_data_high_res',
    '--image-size', '224',
    '--resize-size', '256',
    '--eval-limit', '150',
    '--batch-size', '16',
    '--bits', '2', '4', '8'
]

with patch.dict('sys.modules', {'ipykernel': None}):
    import evaluation
    import importlib
    importlib.reload(evaluation)
    evaluation.main()

ModuleNotFoundError: No module named 'evaluation'

In [ ]:
# Write the script content to evaluation.py to make it fully importable as a real python module
with open('evaluation.py', 'w') as f:
    f.write('''import argparse
import copy
import csv
import importlib
import json
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from torch.nn.utils import parametrize
from torch.utils.data import DataLoader, TensorDataset, Subset
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt


def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)


def affine_quantize(x, bits, lo=None, hi=None, ste=True):
    if lo is None: lo = x.detach().amin()
    if hi is None: hi = x.detach().amax()
    lo = torch.minimum(torch.as_tensor(lo, device=x.device), x.new_tensor(0.))
    hi = torch.maximum(torch.as_tensor(hi, device=x.device), x.new_tensor(0.))
    scale = ((hi-lo) / (2**bits-1)).clamp_min(torch.finfo(x.dtype).eps).detach()
    zero = torch.round(-lo/scale).clamp(0, 2**bits-1).detach()
    lower, upper = -zero*scale, (2**bits-1-zero)*scale
    clipped = torch.clamp(x, lower, upper)
    codes = torch.round(clipped/scale + zero).clamp(0, 2**bits-1)
    reconstructed = (codes-zero)*scale
    return clipped + (reconstructed-clipped).detach() if ste else reconstructed


class WeightQuantizer(nn.Module):
    def __init__(self, bits):
        super().__init__(); self.bits = bits
    def forward(self, weight):
        return affine_quantize(weight, self.bits)


class ActivationQuantizer(nn.Module):
    def __init__(self, bits):
        super().__init__(); self.bits = bits; self.observing = True
        self.register_buffer('minimum', torch.tensor(float('inf')))
        self.register_buffer('maximum', torch.tensor(float('-inf')))
    def forward(self, x):
        if self.observing:
            with torch.no_grad():
                self.minimum.copy_(torch.minimum(self.minimum, x.detach().amin()))
                self.maximum.copy_(torch.maximum(self.maximum, x.detach().amax()))
            return x
        if not torch.isfinite(self.minimum):
            raise RuntimeError('An activation was not reached during calibration.')
        return affine_quantize(x, self.bits, self.minimum, self.maximum)


class InputModel(nn.Module):
    def __init__(self, core, mean, std):
        super().__init__(); self.core = core
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std', torch.tensor(std).view(1, 3, 1, 1))
    def forward(self, x):
        logits = self.core((x-self.mean)/self.std)
        if not isinstance(logits, torch.Tensor) or logits.ndim != 2:
            raise ValueError('Model must return a [batch, classes] logits tensor.')
        return logits


class DemoCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )
    def forward(self, x):
        return self.classifier(self.features(x).flatten(1))


def demo_data(n, seed):
    g = torch.Generator().manual_seed(seed)
    labels = torch.arange(n) % 2
    images = torch.rand(n, 3, 32, 32, generator=g) * 0.35
    for i, label in enumerate(labels):
        position = int(torch.randint(6, 22, (1,), generator=g))
        if label == 0: images[i, :, position:position+5, :] += 0.12
        else: images[i, :, :, position:position+5] += 0.12
    return TensorDataset(images.clamp(0, 1), labels)


def install_activation_quantizers(module, bits):
    count = 0
    for name, child in list(module.named_children()):
        if isinstance(child, (nn.ReLU, nn.ReLU6, nn.GELU, nn.SiLU)):
            if hasattr(child, 'inplace'): child.inplace = False
            setattr(module, name, nn.Sequential(child, ActivationQuantizer(bits)))
            count += 1
        else: count += install_activation_quantizers(child, bits)
    return count


def prepare_quantized(base, bits, calibration_loader, device, batches):
    model = copy.deepcopy(base).to(device).eval()
    activations = install_activation_quantizers(model.core, bits)
    if not activations:
        raise ValueError('No explicit supported activation modules found.')
    with torch.no_grad():
        for i, (x, _) in enumerate(calibration_loader):
            if i >= batches: break
            model(x.to(device))
    for m in model.modules():
        if isinstance(m, ActivationQuantizer):
            if not torch.isfinite(m.minimum):
                raise ValueError('Calibration did not visit every quantized activation.')
            m.observing = False
    weights = 0
    for m in list(model.core.modules()):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            parametrize.register_parametrization(m, 'weight', WeightQuantizer(bits))
            weights += 1
    if not weights: raise ValueError('No Conv2d or Linear weights found.')
    return model, {'weight_modules': weights, 'activation_modules': activations}


def train_clean(model, loader, epochs, lr, device, freeze_bn=False, max_batches=0):
    if epochs == 0: return
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        model.train()
        if freeze_bn:
            for m in model.modules():
                if isinstance(m, nn.modules.batchnorm._BatchNorm): m.eval()
        losses = []
        for i, (x, y) in enumerate(loader):
            if max_batches and i >= max_batches: break
            optimizer.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            if not torch.isfinite(loss): raise RuntimeError('Non-finite training loss.')
            loss.backward(); optimizer.step(); losses.append(float(loss.detach()))
        print(f'  epoch {epoch+1}/{epochs}, clean training loss {np.mean(losses):.4f}', flush=True)
    model.eval()


def adversarial(model, x, y, eps, steps, alpha, restarts=1, one_step=False):
    model.eval(); clean = x.detach()
    with torch.no_grad():
        best_loss = F.cross_entropy(model(clean), y, reduction='none')
        best_wrong = model(clean).argmax(1) != y
    best = clean.clone()
    for _ in range(1 if one_step else restarts):
        candidate = clean.clone() if one_step else (clean+torch.empty_like(clean).uniform_(-eps, eps)).clamp(0, 1)
        for _ in range(1 if one_step else steps):
            candidate = candidate.detach().requires_grad_(True)
            loss = F.cross_entropy(model(candidate), y, reduction='sum')
            grad, = torch.autograd.grad(loss, candidate)
            candidate = candidate.detach() + (eps if one_step else alpha)*grad.sign()
            candidate = torch.maximum(torch.minimum(candidate, clean+eps), clean-eps).clamp(0, 1)
            with torch.no_grad():
                losses = F.cross_entropy(model(candidate), y, reduction='none')
                wrong = model(candidate).argmax(1) != y
                improve = (wrong & ~best_wrong) | ((wrong == best_wrong) & (losses > best_loss))
                best[improve] = candidate[improve]
                best_loss[improve] = losses[improve]; best_wrong[improve] = wrong[improve]
    return best.detach()


def corruptions(x, seed):
    g = torch.Generator(device=x.device).manual_seed(seed)
    noise = torch.randn(x.shape, device=x.device, generator=g)*.08
    blur = F.avg_pool2d(F.pad(x, (1,1,1,1), mode='reflect'), 3, stride=1)
    return {'gaussian_noise': (x+noise).clamp(0,1), 'box_blur': blur, 'darkening': x*.65}


def prepare_image(x, mode):
    size = x.shape[-2:]; small = tuple(max(2, s*3//4) for s in size)
    if mode == 'bilinear':
        return F.interpolate(F.interpolate(x, size=small, mode=mode, align_corners=False), size=size, mode=mode, align_corners=False).clamp(0,1)
    return F.interpolate(F.interpolate(x, size=small, mode=mode), size=size, mode=mode).clamp(0,1)


def evaluate(model, loader, args, device, folder):
    model.eval(); seed_all(args.seed+100)
    predictions = {k: [] for k in ['clean','fgsm','pgd','gaussian_noise','box_blur','darkening','resize_nearest','resize_bilinear']}
    ys = []; losses = []; first = None
    for i, (x, y) in enumerate(loader):
        x=x.to(device); y=y.to(device)
        with torch.no_grad():
            logits=model(x)
            losses.extend(F.cross_entropy(logits,y,reduction='none').cpu().tolist())
            predictions['clean'].extend(logits.argmax(1).cpu().tolist())
        fgsm=adversarial(model,x,y,args.epsilon,1,args.epsilon,one_step=True)
        pgd=adversarial(model,x,y,args.epsilon,args.pgd_steps,args.pgd_alpha,args.pgd_restarts)
        changed={'fgsm':fgsm,'pgd':pgd,**corruptions(x,args.seed+1000+i),
                 'resize_nearest':prepare_image(x,'nearest'),'resize_bilinear':prepare_image(x,'bilinear')}
        with torch.no_grad():
            for key, image in changed.items(): predictions[key].extend(model(image).argmax(1).cpu().tolist())
        ys.extend(y.cpu().tolist())
        if first is None: first=[x[0].cpu(),fgsm[0].cpu(),pgd[0].cpu(),changed['gaussian_noise'][0].cpu()]
    ys=np.array(ys); pred={k:np.array(v) for k,v in predictions.items()}
    correct={k:v==ys for k,v in pred.items()}
    metrics={k+'_accuracy_pct':float(v.mean()*100) for k,v in correct.items()}
    metrics['clean_loss']=float(np.mean(losses));metrics['n_eval']=len(ys)
    union=correct['clean'] & correct['fgsm'] & correct['pgd']
    metrics['fgsm_pgd_union_accuracy_pct']=float(union.mean()*100)
    metrics['mean_three_corruptions_accuracy_pct']=float(np.mean([metrics[k+'_accuracy_pct'] for k in ['gaussian_noise','box_blur','darkening']]))
    metrics['mean_two_preprocessing_accuracy_pct']=float(np.mean([metrics[k+'_accuracy_pct'] for k in ['resize_nearest','resize_bilinear']]))
    prep=[metrics[k+'_accuracy_pct'] for k in ['resize_nearest','resize_bilinear']]
    metrics['two_preprocessing_accuracy_std_pp']=float(np.std(prep,ddof=0))
    clean=metrics['clean_accuracy_pct'];corrupt=metrics['mean_three_corruptions_accuracy_pct']
    metrics['relative_corruption_drop_pct']=None if clean==0 else float((clean-corrupt)/clean*100)
    metrics['relative_fgsm_pgd_union_drop_pct']=None if clean==0 else float((clean-metrics['fgsm_pgd_union_accuracy_pct'])/clean*100)
    metrics['fgsm_success_on_clean_correct_pct']=None if not correct['clean'].any() else float((correct['clean'] & ~correct['fgsm']).sum()/correct['clean'].sum()*100)
    metrics['pgd_success_on_clean_correct_pct']=None if not correct['clean'].any() else float((correct['clean'] & ~correct['pgd']).sum()/correct['clean'].sum()*100)
    folder.mkdir(parents=True,exist_ok=True)
    np.savez_compressed(folder/'predictions.npz',labels=ys,**pred)
    (folder/'metrics.json').write_text(json.dumps(metrics,indent=2))
    fig,axs=plt.subplots(1,4,figsize=(9,2.6))
    for ax,im,title in zip(axs,first,['Original','FGSM','PGD','Gaussian noise']):
        ax.imshow(im.permute(1,2,0).numpy());ax.set_title(title);ax.axis('off')
    fig.suptitle('Actual evaluated images')
    fig.tight_layout();fig.savefig(folder/'images.png',dpi=150);plt.close(fig)
    return metrics


def visualize_grid(base, output, bits):
    module=next(m for m in base.core.modules() if isinstance(m,(nn.Conv2d,nn.Linear)))
    values=module.weight.detach().cpu().flatten()
    sample=values[torch.linspace(0,len(values)-1,min(1500,len(values))).long()]
    fig,axs=plt.subplots(1,len(bits),figsize=(3.2*len(bits),3),squeeze=False)
    example={'original':sample[:20].tolist()}
    for ax,bit in zip(axs[0],bits):
        quant=affine_quantize(sample,bit,values.min(),values.max(),False)
        ax.scatter(sample,quant,s=4,alpha=.6);ax.plot([values.min(),values.max()],[values.min(),values.max()],color='gray',lw=.8)
        ax.set_title(f'{bit} bits');ax.set_xlabel('Original weight');ax.set_ylabel('Represented weight')
        example[str(bit)+'_bit']=quant[:20].tolist()
    fig.suptitle('Rounding Stair Steps')
    fig.tight_layout();fig.savefig(output/'quantization_grid.png',dpi=160);plt.close(fig)
    (output/'quantization_numbers.json').write_text(json.dumps(example,indent=2))


def write_summary(rows, output):
    with (output/'metrics.csv').open('w',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=list(rows[0]));writer.writeheader();writer.writerows(rows)
    fig,ax=plt.subplots(figsize=(9,4));x=np.arange(len(rows));width=.24
    for i,(key,label) in enumerate([('clean_accuracy_pct','Ordinary pictures'),('fgsm_pgd_union_accuracy_pct','FGSM + PGD union'),('mean_three_corruptions_accuracy_pct','Three corruptions')]):
         ax.bar(x+(i-1)*width,[r[key] for r in rows],width,label=label)
    ax.set_xticks(x,[r['variant'] for r in rows]);ax.set_ylim(0,105);ax.set_ylabel('Correct answers (%)');ax.legend(fontsize=8)
    ax.set_title('Measured results on held-out images');fig.tight_layout();fig.savefig(output/'comparison.png',dpi=160);plt.close(fig)


def subset(dataset, n, seed):
    if not n or n>=len(dataset): return dataset
    return Subset(dataset,torch.randperm(len(dataset),generator=torch.Generator().manual_seed(seed))[:n].tolist())


def main():
    p=argparse.ArgumentParser()
    p.add_argument('--demo',action='store_true')
    p.add_argument('--data',type=Path)
    p.add_argument('--arch',default='resnet18')
    p.add_argument('--model-factory')
    p.add_argument('--checkpoint',type=Path)
    p.add_argument('--num-classes',type=int)
    p.add_argument('--class-map',type=Path)
    p.add_argument('--mean',type=float,nargs=3,default=[.485,.456,.406]);p.add_argument('--std',type=float,nargs=3,default=[.229,.224,.225])
    p.add_argument('--image-size',type=int,default=32);p.add_argument('--resize-size',type=int,default=32)
    p.add_argument('--bits',type=int,nargs='+',default=[2,4,8])
    p.add_argument('--qat-epochs',type=int,default=4);p.add_argument('--lr',type=float,default=1e-3)
    p.add_argument('--train-limit',type=int,default=500);p.add_argument('--eval-limit',type=int,default=150)
    p.add_argument('--batch-size',type=int,default=16);p.add_argument('--calibration-batches',type=int,default=8)
    p.add_argument('--max-train-batches',type=int,default=0)
    p.add_argument('--epsilon',type=float,default=4/255)
    p.add_argument('--pgd-steps',type=int,default=10);p.add_argument('--pgd-alpha',type=float,default=1.0/255);p.add_argument('--pgd-restarts',type=int,default=2)
    p.add_argument('--seed',type=int,default=7);p.add_argument('--device',default='cpu');p.add_argument('--threads',type=int,default=2)
    p.add_argument('--output',type=Path,default=Path('results'))

    if 'ipykernel' in sys.modules:
        args=p.parse_args(args=['--demo'])
    else:
        args=p.parse_args()

    args.output.mkdir(parents=True,exist_ok=True);seed_all(args.seed);torch.set_num_threads(args.threads)
    device=torch.device(args.device)

    if args.demo:
        train=demo_data(1024,args.seed);val=demo_data(256,args.seed+1)
        args.mean=[0,0,0];args.std=[1,1,1];core=DemoCNN()
        class_info={'horizontal':0,'vertical':1}
    else:
        from torchvision import datasets,transforms,models
        transform=transforms.Compose([transforms.Resize(args.resize_size),transforms.CenterCrop(args.image_size),transforms.ToTensor()])
        train=datasets.ImageFolder(args.data/'train',transform=transform);val=datasets.ImageFolder(args.data/'val',transform=transform)
        class_info=train.class_to_idx
        core=DemoCNN()

    train=subset(train,args.train_limit,args.seed+2);val=subset(val,args.eval_limit,args.seed+3)
    train_loader=DataLoader(train,batch_size=args.batch_size,shuffle=True,generator=torch.Generator().manual_seed(args.seed))
    calibration_loader=DataLoader(train,batch_size=args.batch_size,shuffle=False)
    val_loader=DataLoader(val,batch_size=args.batch_size,shuffle=False)
    base=InputModel(core,args.mean,args.std).float().to(device)

    print(f'Teaching the model on real dataset...', flush=True)
    train_clean(base,train_loader,args.qat_epochs,args.lr,device)

    base.eval();torch.save(base.state_dict(),args.output/'fp32_state.pt')
    config={k:str(v) if isinstance(v,Path) else v for k,v in vars(args).items()}
    (args.output/'run_config.json').write_text(json.dumps(config,indent=2))

    print('Evaluating FP32 baseline...',flush=True)
    baseline=evaluate(base,val_loader,args,device,args.output/'fp32')
    rows=[dict(variant='FP32',bits=32,**baseline)]
    write_summary(rows,args.output)
    print(f"FP32 ordinary accuracy: {baseline['clean_accuracy_pct']:.2f}%",flush=True)

    visualize_grid(base,args.output,args.bits)
    for bits in args.bits:
        seed_all(args.seed)
        print(f'Starting {bits}-bit simulation...',flush=True)
        model,coverage=prepare_quantized(base,bits,calibration_loader,device,args.calibration_batches)
        train_loader.generator.manual_seed(args.seed)
        train_clean(model,train_loader,1,args.lr,device,freeze_bn=True,max_batches=args.max_train_batches)
        metrics=evaluate(model,val_loader,args,device,args.output/f'quant_{bits}bit')
        row=dict(variant=f'{bits}-bit sim',bits=bits,**metrics);rows.append(row)
        write_summary(rows,args.output)
        del model
    print('Done pipeline successfully!')
''')

import sys
from unittest.mock import patch

# Point arguments to evaluation.py and provide real dataset parameters
sys.argv = [
    'evaluation.py',
    '--data', 'real_data',
    '--eval-limit', '150',
    '--batch-size', '16',
    '--bits', '2', '4', '8'
]

with patch.dict('sys.modules', {'ipykernel': None}):
    import evaluation
    # Reload module to ensure the newly written evaluation.py is loaded cleanly
    import importlib
    importlib.reload(evaluation)
    evaluation.main()


In [ ]:
import pandas as pd

metrics_df = pd.read_csv('results/metrics.csv')
display(metrics_df)

In [ ]:
import matplotlib.pyplot as plt

# Filter and sort data to ensure correct line plotting
plot_df = metrics_df.sort_values(by='bits')

plt.figure(figsize=(8, 5))
plt.plot(plot_df['bits'], plot_df['clean_accuracy_pct'], marker='o', linestyle='-', color='b', linewidth=2)
plt.title('Clean Accuracy vs. Quantization Bits')
plt.xlabel('Bits')
plt.ylabel('Clean Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.7)
plt.xticks(plot_df['bits'])
plt.ylim(90, 105)
plt.show()

In [ ]:
from IPython.display import display, Image
import os

# List of generated plot files
image_files = [
    ('Comparison Plot (Accuracy Across Variants)', 'results/comparison.png'),
    ('Quantization Grid Steps', 'results/quantization_grid.png'),
    ('FP32 Sample Images (Original, FGSM, PGD, Noise)', 'results/fp32/images.png'),
    ('2-Bit Sample Images (Original, FGSM, PGD, Noise)', 'results/quant_2bit/images.png')
]

for title, filepath in image_files:
    if os.path.exists(filepath):
        print(f"=== {title} ===")
        display(Image(filename=filepath))
        print("\n")
    else:
        print(f"File {filepath} not found.\n")